# Star Graph Pipeline on Kaggle

Runs enrichment + deep research using a local llama.cpp model (no NVIDIA NIM).
Pushes results back to GitHub.

This notebook runs automatically when pushed via `kaggle kernels push`.
For manual use: run cells top-to-bottom.

**First-time setup:** Set GH_PAT in Kaggle Secrets (left sidebar) with `repo` scope.

In [ ]:
# --- clone both repos ---------------------------------------------------
import subprocess, sys, os, time

for d in ['/kaggle/working/kms', '/kaggle/working/star-graph']:
    subprocess.run(['rm', '-rf', d], capture_output=True)

subprocess.run(['git', 'clone', '--depth', '1',
    'https://github.com/Meru143/kaggle-model-server.git', '/kaggle/working/kms'],
    check=True, timeout=120)
subprocess.run(['git', 'clone', '--depth', '1',
    'https://github.com/Meru143/star-graph.git', '/kaggle/working/star-graph'],
    check=True, timeout=120)

sys.path.insert(0, '/kaggle/working/kms')
sys.path.insert(0, '/kaggle/working/star-graph/kaggle')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface_hub', 'requests', 'networkx', 'numpy', 'sentence-transformers'],
    check=True, timeout=300)
print('Setup complete')

In [ ]:
# --- imports + secrets --------------------------------------------------
import importlib
for _m in ('model_registry', 'harness'):
    if _m in sys.modules:
        if _m == 'harness':
            try: sys.modules['harness'].stop()
            except Exception: pass
        importlib.reload(sys.modules[_m])

from model_registry import MODELS
from harness import run, stop, harvest_cache
from star_graph_kaggle import run_pipeline

# GH_PAT for pushing to GitHub
try:
    import kaggle_secrets
    os.environ.setdefault('GH_PAT', kaggle_secrets.UserSecretsClient().get_secret('GH_PAT'))
    print('GH_PAT loaded')
except Exception:
    print('WARNING: GH_PAT not set. Pipeline will run but cannot push.')

print(f'Models: {list(MODELS.keys())}')

In [ ]:
# --- boot the local LLM -------------------------------------------------
# Nanbeige4.2-3B Q4_K_M: 2.5GB on GPU 0, temp 0.0 for JSON extraction
# First run: downloads GGUF + builds llama.cpp (~15 min)
# Cached run: ~2 min (after harvest_cache done once)

url = run('owao/Nanbeige4.2-3B-GGUF', MODELS, quant='Q4_K_M', ctx=4096)
print(f'Model ready at {url}')

In [ ]:
# --- run the pipeline (no NVIDIA calls, all local) ----------------------
run_pipeline(limit=None)
print('Pipeline complete')

In [ ]:
# --- cache binaries for faster next boot --------------------------------
try:
    harvest_cache()
    print('Cache harvested')
except Exception as e:
    print(f'Cache harvest skipped: {e}')

In [ ]:
# --- clean shutdown -----------------------------------------------------
stop()
print('Model stopped. Session will auto-terminate.')